# Zero-TVM throughput bench — Colab T4

**First: Runtime → Change runtime type → T4 GPU.**

> **Heads-up:** Colab ships the CUDA driver but usually **not** the GL/Vulkan userspace Chrome's WebGPU needs, so cell 1 often reports `llvmpipe` (CPU) instead of `Tesla T4`. If it does, the bench will stop itself (software numbers are meaningless) — use **Vast.ai** instead (rent a T4, set `NVIDIA_DRIVER_CAPABILITIES=all`).

In [ ]:
# 1) Node 22 + Chrome/Vulkan deps, then try to expose the T4 to Vulkan
!nvidia-smi --query-gpu=name,driver_version --format=csv,noheader
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1
!apt-get -qq install -y nodejs vulkan-tools libvulkan1 xvfb libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libgbm1 libasound2 libxcomposite1 libxdamage1 libxrandr2 libxkbcommon0 libxfixes3 libdrm2 libpango-1.0-0 libcairo2 libatspi2.0-0 fonts-liberation > /dev/null
!mkdir -p /usr/share/vulkan/icd.d
!printf '{"file_format_version":"1.0.0","ICD":{"library_path":"libGLX_nvidia.so.0","api_version":"1.3.277"}}' > /usr/share/vulkan/icd.d/nvidia_icd.json
print('--- NVIDIA Vulkan lib present?', end=' ')
!ls /usr/lib/x86_64-linux-gnu/libGLX_nvidia.so.0 2>/dev/null || echo 'NO -> Colab cannot do WebGPU, use Vast.ai'
print('--- Vulkan device (want Tesla T4, not llvmpipe): ---')
!vulkaninfo --summary 2>/dev/null | grep -E 'deviceName|driverName' || echo 'NO VULKAN DEVICE'

In [ ]:
# 2) Clone + run the bench. Aborts on its own if cell 1 showed llvmpipe (software).
#    First run downloads ~2 GB of Phi-3 weights.
!pip -q install -U huggingface_hub
!git clone -q https://github.com/abgnydn/zero-tvm.git
!cd zero-tvm && npm ci --silent && BENCH_HEADLESS=new bash bench/cloud-bench.sh

## Done

Copy the printed `results.json` into your repo at `bench/results.json`, then run `npm run bench:sync -- --write` locally (no GPU) to update BENCH.md + the bench page.